In [ ]:
#DONE: use B and E instead of A and B for begin and end points segments

In [ ]:
# Settings

from pathlib import Path
import pandas as pd
import numpy as np

latMeter = 0.0000089988659514815        # 1 meter expressed in latitude (works with area province Utrecht)
lonMeter = 0.0000146436902975532        # 1 meter expressed in longitude (works with area province Utrecht)
latMin = 51.858631                      # smallest latitude province Utrecht
lonMin = 4.794457                       # smallest longitude province Utrecht
distance = 20                           # distance (meters) from route to attribute to route(segment)

In [ ]:
# From rapportage.ipynb
def get_period_info(period_spec, year, quarter, month):
    """Return an id-string and the months for the chosen period."""

    quarters = {
        'Q1': [1, 2, 3], 
        'Q2': [4, 5, 6], 
        'Q3': [7, 8, 9], 
        'Q4': [10, 11, 12], 
    }
    periods = {
        'jaar': {
            'period_id': f'{year}',
            'months': list(range(1, 13)),
        },
        'kwartaal': {
            'period_id': f'{year}_{quarter}',
            'months': quarters[quarter],
        },
        'maand': {
            'period_id': f'{year}_{month:02d}',
            'months': [month],
        }
    }
    period_id = periods[period_spec]['period_id']
    months = periods[period_spec]['months']

    return period_id, months

In [ ]:
# From rapportage.ipynb - using 'mcu_gegevens' instead of 'api_gegevens'
# File system settings.
prefix = 'mcu_gegevens'         # de prefix voor de csv-datafiles; default format <prefix>_<yyyy>_<mm>.csv
data_directory = Path(
    Path('~').expanduser(),
    'Documents',
    'MCUdataclub',
    'Snuffelfiets_data',
    'rapportage',
    )                           # het pad naar de folder met de data; dit wordt ook de parent folder van de output
Path.mkdir(data_directory, parents=True, exist_ok=True)
print(data_directory)
# Date range settings.
period_spec = 'kwartaal'        # de periode waar het rapport over gaat: 'maand', 'kwartaal' of 'jaar'?
year = 2024                     # het jaar waar het rapport over gaat
quarter = 'Q3'                  # het kwartaal waar het rapport over gaat: 'Q1', 'Q2', 'Q3', 'Q4'
month = 6                       # de maand waar het rapport over gaat: 1, 2, 3, ..., 10, 11, 12

# Directories.
period_id, months = get_period_info(period_spec, year, quarter, month)
output_directory = Path(data_directory, period_id)
output_directory.mkdir(parents=True, exist_ok=True)

#print(f'Analysing {period_spec} {period_id}; writing output to {output_directory}.')

In [ ]:
# From rapportage.ipynb
# TODO implement method to read all data files within data range
# TODO modify except FileNotFoundError
# Read the data for the selected period (in monthly chunks)
#'''
dfs = []

for m in months:

    filename = f'{prefix}_{year}-{m:02d}.csv'
    p = Path(data_directory, filename)
    print(p)
    try:

        # try to read the data from saved csv's
        df = pd.read_csv(p)

    except FileNotFoundError:

        # download the data if the file does not exist
        inlezen.monthly_csv_dump(api_key, year, m, data_directory, prefix)

        df = pd.read_csv(p)

    dfs.append(df)

df = pd.concat(dfs, ignore_index=True) #bug fixed by ignore_index=True

print(f'Read {df.shape[0]} measurements.')
#'''

In [ ]:
'''
# Test: read all data from D:\Snuffelfietsdata\mcu_bewerkt

from pathlib import Path
from pathlib import PureWindowsPath

dfs = []
prefix = "mcu_gegevens"
data_directory2 = PureWindowsPath('D:\Snuffelfietsdata\mcu_bewerkt')

for year in [2021, 2022, 2023, 2024]:                           #[2021, 2022, 2023, 2024]
    for month in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]:
        filename = f'{prefix}_{year}-{month:02d}.csv'
        p = Path(data_directory2, filename)
        #print(p)
        df = pd.read_csv(p)
        dfs.append(df)

df = pd.concat(dfs, ignore_index=True) #bug fixed by ignore_index=True
print(f'Read {df.shape[0]} measurements.')
'''

In [ ]:
#add columns yLat and xLon 

#latMin = 0
#lonMin = 0
df['yLat'] = (df['latitude'] - latMin) / latMeter
df['xLon'] = (df['longitude'] - lonMin) / lonMeter
df['yLat'] = (df['latitude'] - latMin) / latMeter
df['xLon'] = (df['longitude'] - lonMin) / lonMeter


# make rit_id unique
df['rit_id'] = df['rit_id'] + df['year'] * 1000000 + df['month'] * 10000

print(f'Read {df.shape[0]} measurements.')

In [ ]:
# Import route points

filename = 'route1.csv'
p = Path(data_directory, filename)
dfRoutePoints = pd.read_csv(p)

Per route point

Per segment |BE|

In [ ]:
#test route
latPoint = [52.0801829081972, 52.0816519526292, 52.0849999945306, 52.0780696744497, 52.0762778003675]
lonPoint = [5.10326380258989, 5.10482561813942, 5.10875574551401, 5.11513102441433, 5.12067050761371]

#latPoint2 = dfRoutePoints['latitude']
#lonPoint2 = dfRoutePoints['longitude']
#print(latPoint2)

# TODO create loop through dfRoutePoints
# TODO get xB, xE, yB, yE from dfRoutePoints    OR    use dfRoutePoints['yLat'] etc. directly 
# 0-1, 1-2 
yB = (latPoint[1] - latMin) / latMeter
yE = (latPoint[2] - latMin) / latMeter
xB = (lonPoint[1] - lonMin) / lonMeter
xE = (lonPoint[2] - lonMin) / lonMeter

dist = 0
dist = distance         # dist increases filter range by distance

'''
# filter range 
dfSegment = df
dfSegment = dfSegment[dfSegment['yLat'].between(min(yB, yE) - dist, max(yB, yE) + dist)]
dfSegment = dfSegment[dfSegment['xLon'].between(min(xB, xE) - dist, max(xB, xE) + dist)]

# how many points in filter range
print(f'Filtered {dfSegment.shape[0]} possible measurements for segment #.')

# export filter range
filename = "dfSegment-filter.csv"
p = Path(data_directory, filename)
dfSegment.to_csv(p, index=False)

# test whether filter range saves time or not
#dfSegment = df
df = dfSegment
'''

'''# test data
xB = 700
yB = 900
xE = 3000
yE = 1500
'''


In [ ]:
# Extend segment with distance from end point E

BE = np.sqrt( 
    np.square(xE - xB) 
    + np.square(yE - yB) )

yE = (yE - yB) * distance / BE + yE
xE = (xE - xB) * distance / BE + xE

In [ ]:
# Calculate sides (BE, MB, ME) and angles (aMB, aME, aBE) to calculate distance M to side BE
# see 'hoeken voorbeeld.xlsx'

BE = np.sqrt( 
    np.square(xE - xB) 
    + np.square(yE - yB) ) 

df['MB'] = np.sqrt( 
    np.square(df['xLon'] - xB) 
    + np.square(df['yLat'] - yB) ) 

df['ME'] = np.sqrt(
    np.square(df['xLon'] - xE) 
    + np.square(df['yLat'] - yE) ) 

df['aBE'] = np.acos(                                                    # angle aBE is not used at all?
    ( np.square(df['MB']) + np.square(df['ME']) - np.square(BE) ) 
    / (2 * df['MB'] * df['ME']) ) 

df['aMB'] = np.acos( 
    ( np.square(df['ME']) + np.square(BE) - np.square(df['MB']) ) 
    / (2 * BE * df['ME']) ) 

df['aME'] = np.acos( 
    ( np.square(df['MB']) + np.square(BE) - np.square(df['ME']) ) 
    / (2 * BE * df['MB']) ) 

df['M-BE'] = np.sin(df['aME']) * df['MB'] 

In [ ]:
#TODO fill in Route column
# column route# = segment# if P-BE<distance AND max(aPB, aPE)<PI/2

In [ ]:
dfWithin = df[df['M-BE'].between( - distance, distance)]
dfWithin = dfWithin[dfWithin['aMB'].between( 0, np.pi/2)]
dfWithin = dfWithin[dfWithin['aME'].between( 0, np.pi/2)]

#dfOutside = df[~df['M-BE'].between( - distance, distance)]
#dfOutside = dfOutside[~dfOutside['aMB'].between( 0, np.pi/2)]
#dfOutside = dfOutside[~dfOutside['aME'].between( 0, np.pi/2)]

In [ ]:
print(f'Calculated M-BE for {df.shape[0]} measurements.')
print(df['M-BE'].head(10))

# filter measurements within chosen distance from route segment
#dfWithin = df[df['M-BE'].between( - distance, distance)]
#dfOutside = df[~df['M-BE'].between( - distance, distance)]

#TODO export with date and timestamp to not overwrite previous files

print(f'Found {dfWithin.shape[0]} measurements within {distance} meter of segment #.')
print(dfWithin['M-BE'].head(10))

# export 
filename = "dfWithin.csv"
p = Path(data_directory, filename)
dfWithin.to_csv(p, index=False)

#filename = "dfOutside.csv"
#p = Path(data_directory, filename)
#dfOutside.to_csv(p, index=False)

#TODO for testing, I want three csv exports: 
#   1) points within filter range AND within distance from segment
#   2) points within filter range AND NOT within distance from segment
#   maybe 3) limited points surrounding filter range
#   so I can visually check in Google Earth if the outcome seems correct...

#TODO what happens to M-BE when point isn't perpendicular to segment? do I need to account for those points?

In [ ]:
#print(dfr.iloc[1])
#print(dfr.iloc[1]['latitude'])